In [ ]:
BAT_CALL_JSON = "BD-A-99-003-analysis.json"

In [ ]:
%run pathutils.ipynb
%run export.ipynb

# Heterodyne Pulse Analysis

This notebook is the heterodyne counterpart to the time-expansion pulse analysis notebook. It focuses on timing and sequence structure, which are preserved in heterodyne recordings, and avoids spectral interpretation.

It now also includes a simple rule-based behavioural classification at sequence level. Each detected sequence is labelled from timing structure alone, using fields such as `sequence_id`, `pulse_in_sequence`, `real_ipi_next_ms`, `local_ipi_median_ms`, `local_ipi_min_ms`, and `is_terminal_dense_run`.

## Sequence-level behavioural labels

- **feeding_pass**: sequence contains a terminal dense run
- **approach_only**: sequence shows substantial IPI compression but no terminal dense run
- **search_or_transit**: sequence remains comparatively even in timing, with no strong terminal compression
- **insufficient_data**: too few usable pulses for a sensible classification


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Construct the full path to the JSON file, which is expected to be in the "data" folder
json_file = (Path("..") / "data" / BAT_CALL_JSON).resolve()

# Load the analysis JSON
with open(json_file, "r") as f:
    data = json.load(f)

# Check this is a heterodyne analysis
if data["analysis_mode"] != "heterodyne":
    message = f"Expected analysis_mode='heterodyne', got {data['analysis_mode']!r}. Aborting notebook."
    raise ValueError(message)

# Load pulses and sequences
pulses = data.get("pulses", [])
sequences = data.get("sequences", [])

if not pulses:
    raise ValueError("No pulses found in the heterodyne analysis JSON.")

df = pd.json_normalize(pulses)
seq_df = pd.json_normalize(sequences) if sequences else pd.DataFrame()

# Ensure a plotting index exists
if "pulse_index" not in df.columns:
    if "index" in df.columns:
        df["pulse_index"] = df["index"]
    else:
        df["pulse_index"] = range(1, len(df) + 1)

# Fallback if no explicit sequence_id was stored
if "sequence_id" not in df.columns:
    df["sequence_id"] = 1
    seq_df = pd.DataFrame([{
        "sequence_id": 1,
        "start_time_s": df["real_start_time_s"].min(),
        "end_time_s": df["real_end_time_s"].max(),
        "pulse_count": len(df),
        "duration_s": df["real_end_time_s"].max() - df["real_start_time_s"].min(),
        "mean_ipi_ms": df["real_ipi_next_ms"].dropna().mean(),
        "median_ipi_ms": df["real_ipi_next_ms"].dropna().median(),
        "min_ipi_ms": df["real_ipi_next_ms"].dropna().min(),
        "has_terminal_dense_run": bool(df.get("is_terminal_dense_run", pd.Series([False] * len(df))).max()),
        "terminal_dense_pulse_count": int(df.get("is_terminal_dense_run", pd.Series([False] * len(df))).sum()),
    }])

display(df.head())
display(seq_df)


# Inter-Pulse Interval (IPI) Profile

This chart shows the interval to the next pulse, in real milliseconds, through each detected sequence.

## Purpose
- show pacing through the sequence
- reveal tightening pulse spacing
- highlight dense late-stage clusters where present

## Interpretation
- larger IPI values usually indicate more widely spaced search-phase calls
- falling IPI suggests tightening temporal structure
- very short IPI values can indicate a dense terminal section

## Notes
For heterodyne recordings this is one of the most trustworthy behavioural views, because timing is preserved even though original frequency is not.


In [ ]:
import matplotlib.pyplot as plt

def plot_ipi_profile(seq_pulses, sequence_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    plt.plot(
        seq_pulses["pulse_in_sequence"] if "pulse_in_sequence" in seq_pulses.columns else seq_pulses["pulse_index"],
        seq_pulses["real_ipi_next_ms"],
        marker="o",
        label="IPI to next pulse"
    )

    if "local_ipi_median_ms" in seq_pulses.columns:
        plt.plot(
            seq_pulses["pulse_in_sequence"] if "pulse_in_sequence" in seq_pulses.columns else seq_pulses["pulse_index"],
            seq_pulses["local_ipi_median_ms"],
            marker="o",
            linestyle="--",
            label="Local IPI median"
        )

    if "is_terminal_dense_run" in seq_pulses.columns:
        dense = seq_pulses[seq_pulses["is_terminal_dense_run"] == True]
        if not dense.empty:
            plt.scatter(
                dense["pulse_in_sequence"] if "pulse_in_sequence" in dense.columns else dense["pulse_index"],
                dense["real_ipi_next_ms"],
                label="Terminal dense run"
            )

    plt.xlabel("Pulse number within sequence")
    plt.ylabel("IPI (ms, real time)")
    plt.title(f"IPI Profile for {BAT_CALL_JSON}, Sequence {sequence_id}")
    plt.grid(True)
    plt.legend()

    export_chart(export_folder, f"{export_file_stem}-seq-{str(sequence_id).zfill(3)}-ipi", "png")
    plt.show()


# Local IPI Context

This chart compares each pulse's local median and local minimum IPI values.

## Purpose
- summarise the local timing neighbourhood around each pulse
- show whether a sequence contains local compression
- support interpretation of dense runs without relying on frequency

## Interpretation
- falling local median suggests overall tightening of the sequence
- low local minimum values indicate locally compressed pulse spacing
- divergence between local median and local minimum can show uneven structure


In [ ]:
def plot_local_ipi_context(seq_pulses, sequence_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    x = seq_pulses["pulse_in_sequence"] if "pulse_in_sequence" in seq_pulses.columns else seq_pulses["pulse_index"]

    if "local_ipi_median_ms" in seq_pulses.columns:
        plt.plot(x, seq_pulses["local_ipi_median_ms"], marker="o", label="Local IPI median")

    if "local_ipi_min_ms" in seq_pulses.columns:
        plt.plot(x, seq_pulses["local_ipi_min_ms"], marker="o", linestyle="--", label="Local IPI minimum")

    plt.xlabel("Pulse number within sequence")
    plt.ylabel("IPI (ms)")
    plt.title(f"Local IPI Context for {BAT_CALL_JSON}, Sequence {sequence_id}")
    plt.grid(True)
    plt.legend()

    export_chart(export_folder, f"{export_file_stem}-seq-{str(sequence_id).zfill(3)}-local-ipi", "png")
    plt.show()


# Pulse Duration

This chart shows measured pulse duration in real milliseconds through each sequence.

## Purpose
- show how measured pulse length changes through the call train
- complement IPI structure with a simple within-sequence shape measure

## Notes
These are analyser-derived pulse-region durations. They remain useful in heterodyne work, but should still be read as segmentation-based measurements rather than exact biological pulse widths.


In [ ]:
def plot_pulse_duration(seq_pulses, sequence_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    plt.plot(
        seq_pulses["pulse_in_sequence"] if "pulse_in_sequence" in seq_pulses.columns else seq_pulses["pulse_index"],
        seq_pulses["real_duration_ms"],
        marker="o"
    )

    plt.xlabel("Pulse number within sequence")
    plt.ylabel("Pulse duration (ms, real)")
    plt.title(f"Pulse Duration for {BAT_CALL_JSON}, Sequence {sequence_id}")
    plt.grid(True)

    export_chart(export_folder, f"{export_file_stem}-seq-{str(sequence_id).zfill(3)}-duration", "png")
    plt.show()


# Sequence Timeline

This chart places each pulse on the real-time recording timeline.

## Purpose
- show how sequences are arranged across the recording
- make gaps, clustering, and dense end-runs easy to see
- provide a structural overview without using frequency or amplitude

## Interpretation
- long gaps separate sequences or bouts
- tightly packed pulse bars indicate temporal compression
- dense-run labels can be checked visually against the timeline


In [ ]:
def plot_sequence_timeline(seq_pulses, sequence_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, max(4, len(seq_pulses) * 0.18)))

    y = list(range(1, len(seq_pulses) + 1))
    starts = seq_pulses["real_start_time_s"].tolist()
    ends = seq_pulses["real_end_time_s"].tolist()

    for yi, start, end in zip(y, starts, ends):
        plt.hlines(yi, start, end, linewidth=2)

    if "is_terminal_dense_run" in seq_pulses.columns:
        dense = seq_pulses[seq_pulses["is_terminal_dense_run"] == True]
        if not dense.empty:
            dense_y = [seq_pulses.index.get_loc(idx) + 1 for idx in dense.index]
            plt.scatter(dense["real_peak_time_s"], dense_y, label="Terminal dense run")

    plt.xlabel("Recording time (s)")
    plt.ylabel("Pulse number within sequence")
    plt.title(f"Sequence Timeline for {BAT_CALL_JSON}, Sequence {sequence_id}")
    plt.grid(True)
    if "is_terminal_dense_run" in seq_pulses.columns and seq_pulses["is_terminal_dense_run"].any():
        plt.legend()

    export_chart(export_folder, f"{export_file_stem}-seq-{str(sequence_id).zfill(3)}-timeline", "png")
    plt.show()


# Sequence Classification

This section applies a simple rule-based behavioural classifier to each sequence. It uses timing structure only.

## Logic

- terminal dense run present → `feeding_pass`
- strong or moderate IPI compression without dense run → `approach_only`
- otherwise → `search_or_transit`


In [ ]:

def classify_sequence(seq_pulses):
    seq_pulses = seq_pulses.copy().sort_values(
        "pulse_in_sequence" if "pulse_in_sequence" in seq_pulses.columns else "pulse_index"
    )

    ipi = seq_pulses["real_ipi_next_ms"].dropna()
    pulse_count = len(seq_pulses)

    has_dense_run = False
    if "is_terminal_dense_run" in seq_pulses.columns:
        has_dense_run = bool(seq_pulses["is_terminal_dense_run"].fillna(False).any())

    first_ipi = ipi.iloc[0] if len(ipi) >= 1 else None
    median_ipi = ipi.median() if len(ipi) >= 1 else None
    min_ipi = ipi.min() if len(ipi) >= 1 else None

    early_window = ipi.head(max(3, len(ipi) // 4)) if len(ipi) else ipi
    late_window = ipi.tail(max(3, len(ipi) // 4)) if len(ipi) else ipi
    early_median = early_window.median() if len(early_window) else None
    late_median = late_window.median() if len(late_window) else None

    compression_ratio = None
    if early_median and late_median is not None and early_median > 0:
        compression_ratio = late_median / early_median

    terminal_fraction = None
    if median_ipi is not None and median_ipi > 0 and min_ipi is not None:
        terminal_fraction = min_ipi / median_ipi

    if pulse_count < 3 or len(ipi) < 2:
        label = "insufficient_data"
        reason = "Too few pulses/IPIs for timing-based classification"
    elif has_dense_run:
        label = "feeding_pass"
        reason = "Terminal dense run detected"
    elif (
        compression_ratio is not None
        and terminal_fraction is not None
        and compression_ratio <= 0.75
        and terminal_fraction <= 0.70
    ):
        label = "approach_only"
        reason = "Strong late-sequence IPI compression without terminal dense run"
    elif (
        terminal_fraction is not None
        and terminal_fraction <= 0.80
    ):
        label = "approach_only"
        reason = "Moderate IPI compression without terminal dense run"
    else:
        label = "search_or_transit"
        reason = "No strong timing compression or terminal dense run"

    return {
        "behaviour_label": label,
        "classification_reason": reason,
        "classification_first_ipi_ms": first_ipi,
        "classification_median_ipi_ms": median_ipi,
        "classification_min_ipi_ms": min_ipi,
        "classification_early_median_ipi_ms": early_median,
        "classification_late_median_ipi_ms": late_median,
        "classification_compression_ratio": compression_ratio,
        "classification_terminal_fraction": terminal_fraction,
        "classification_has_terminal_dense_run": has_dense_run,
    }


In [ ]:

# Build sequence-level summaries
sequence_summary = (
    df.groupby("sequence_id")
      .agg(
          pulse_count=("pulse_index", "count"),
          start_s=("real_start_time_s", "min"),
          end_s=("real_end_time_s", "max"),
          duration_s=("real_end_time_s", lambda s: s.max() - s.min()),
          mean_ipi_ms=("real_ipi_next_ms", "mean"),
          median_ipi_ms=("real_ipi_next_ms", "median"),
          min_ipi_ms=("real_ipi_next_ms", "min"),
          mean_duration_ms=("real_duration_ms", "mean"),
          median_duration_ms=("real_duration_ms", "median"),
          has_terminal_dense_run=("is_terminal_dense_run", "max") if "is_terminal_dense_run" in df.columns else ("pulse_index", lambda s: False),
          dense_run_pulse_count=("is_terminal_dense_run", "sum") if "is_terminal_dense_run" in df.columns else ("pulse_index", lambda s: 0),
      )
      .reset_index()
)

# Add local-IPI summary fields where available
if "local_ipi_median_ms" in df.columns:
    local_summary = (
        df.groupby("sequence_id")
          .agg(
              median_local_ipi_ms=("local_ipi_median_ms", "median"),
              min_local_ipi_ms=("local_ipi_min_ms", "min"),
          )
          .reset_index()
    )
    sequence_summary = sequence_summary.merge(local_summary, on="sequence_id", how="left")

# Add behavioural classification
classification_df = pd.DataFrame([
    {"sequence_id": sequence_id, **classify_sequence(seq_pulses)}
    for sequence_id, seq_pulses in df.groupby("sequence_id")
])
sequence_summary = sequence_summary.merge(classification_df, on="sequence_id", how="left")

display(sequence_summary)


In [ ]:

from pathlib import Path

# Construct the export file stem and folder
export_file_stem = Path(BAT_CALL_JSON).stem
export_folder = Path(get_project_root_folder()) / "exported"
export_folder.mkdir(parents=True, exist_ok=True)

# Initialise the export dictionary
export_dict = {
    "Sequences": sequence_summary
}

# Include raw sequence metadata if present
if not seq_df.empty:
    export_dict["Sequence Metadata"] = seq_df

classification_overview = sequence_summary[[
    "sequence_id",
    "pulse_count",
    "behaviour_label",
    "classification_reason",
    "classification_compression_ratio",
    "classification_terminal_fraction",
    "has_terminal_dense_run",
    "dense_run_pulse_count",
]].copy()
export_dict["Classification Overview"] = classification_overview

# Iterate over each sequence
for sequence_id, seq_pulses in df.groupby("sequence_id"):
    seq_pulses = seq_pulses.copy()
    export_dict[f"Sequence {str(sequence_id).zfill(3)}"] = seq_pulses

    plot_ipi_profile(seq_pulses, sequence_id, export_folder, export_file_stem)
    plot_local_ipi_context(seq_pulses, sequence_id, export_folder, export_file_stem)
    plot_pulse_duration(seq_pulses, sequence_id, export_folder, export_file_stem)
    plot_sequence_timeline(seq_pulses, sequence_id, export_folder, export_file_stem)

# Export the data
export_to_spreadsheet(export_folder, f"{export_file_stem}.xlsx", export_dict)
